<a href="https://colab.research.google.com/github/blondedvicky/PROJET-PENDU/blob/main/Copy_of_interface_fonctions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from fonctions import charger_mots_par_difficulte, calculer_score, sauvegarder_score
import tkinter as tk
from tkinter import messagebox, simpledialog # Ajout de simpledialog ici
import random

# Difficulté par défaut
diff_choisie = "Facile"

def demarrer_partie():
    global diff_choisie

    # 1. Utilisation de ta fonction moteur (On lui passe la difficulté actuelle)
    liste_mots = charger_mots_par_difficulte(diff_choisie)

    if not liste_mots:
        messagebox.showerror("Erreur", "Impossible de charger les mots. Vérifiez le dossier 'files/'.")
        return

    # 2. Choix du mot
    mot_secret = random.choice(liste_mots)

    # --- Création de l'interface de jeu ---
    fenetre_de_jeu = tk.Toplevel(root)
    fenetre_de_jeu.title("Jeu du pendu")
    fenetre_de_jeu.geometry("600x400")

    canvas = tk.Canvas(fenetre_de_jeu, width=200, height=250, bg="white")
    canvas.pack()

    mot_secret_affiche = ["_"] * len(mot_secret)
    label_mot = tk.Label(fenetre_de_jeu, text=" ".join(mot_secret_affiche), font=("Arial", 24))
    label_mot.pack(pady=20)

    label_commentaires = tk.Label(fenetre_de_jeu, text="", font = ("Arial", 14))
    label_commentaires.pack(pady=10)

    entree = tk.Entry(fenetre_de_jeu, font=("Arial", 14), width=5)
    entree.pack(pady=5)

    nb_erreurs = 0
    lettres_devinees = []

    def valider(event=None): # 'event' permet de gérer la touche Entrée
        nonlocal nb_erreurs
        nonlocal lettres_devinees

        lettre = entree.get().strip().lower()
        entree.delete(0, tk.END)

        if len(lettre) != 1 or not lettre.isalpha():
            label_commentaires.config(text="Veuillez entrer une seule lettre.")
            return

        if lettre in lettres_devinees:
            label_commentaires.config(text="Déjà proposé !")
            return

        lettres_devinees.append(lettre)

        if lettre in mot_secret:
            for i in range(len(mot_secret)):
                if mot_secret[i] == lettre:
                    mot_secret_affiche[i] = lettre

            label_commentaires.config(text="Bien joué !")
            label_mot.config(text=" ".join(mot_secret_affiche))

            # --- VICTOIRE ---
            if "_" not in mot_secret_affiche:
                label_commentaires.config(text="Félicitations !")
                entree.config(state="disabled")

                # Utilisation de tes fonctions moteur
                score_final = calculer_score(mot_secret, nb_erreurs)
                pseudo = simpledialog.askstring("Victoire", f"Score : {score_final}\nEntrez votre pseudo :")

                if pseudo:
                    sauvegarder_score(pseudo, score_final, "scores.txt")
                return

        else:
            # --- ERREUR ---
            nb_erreurs += 1
            dessiner_pendu(canvas, nb_erreurs)
            label_commentaires.config(text=f"Raté ! {7 - nb_erreurs} chances restantes.")

            if nb_erreurs >= 7:
                label_commentaires.config(text=f"Perdu ! Le mot était : {mot_secret}")
                entree.config(state="disabled")
                return

        label_mot.config(text=" ".join(mot_secret_affiche))

    def dessiner_pendu(canvas, nb_erreurs):
        if nb_erreurs == 1: canvas.create_line(20, 230, 180, 230, width=3)
        elif nb_erreurs == 2: canvas.create_line(60, 230, 60, 20, width=3)
        elif nb_erreurs == 3:
            canvas.create_line(60, 20, 140, 20, width=3)
            canvas.create_line(140, 20, 140, 50, width=2)
        elif nb_erreurs == 4: canvas.create_oval(120, 50, 160, 90, width=2)
        elif nb_erreurs == 5: canvas.create_line(140, 90, 140, 150, width=2)
        elif nb_erreurs == 6:
            canvas.create_line(140, 100, 110, 130, width=2)
            canvas.create_line(140, 100, 170, 130, width=2)
        elif nb_erreurs == 7:
            canvas.create_line(140, 150, 110, 190, width=2)
            canvas.create_line(140, 150, 170, 190, width=2)
            # Visage X_X
            canvas.create_line(130, 65, 135, 75, fill="red"); canvas.create_line(135, 65, 130, 75, fill="red")
            canvas.create_line(145, 65, 150, 75, fill="red"); canvas.create_line(150, 65, 145, 75, fill="red")

    entree.bind("<Return>", valider)
    entree.focus()

# --- Les autres fonctions (Score, Aide, Difficulté) restent identiques ---

def afficher_score():
    try:
        with open("scores.txt", "r", encoding="utf-8") as f:
            historique = f.read()
        messagebox.showinfo("Meilleurs Scores", historique if historique else "Aucun score.")
    except FileNotFoundError:
        messagebox.showerror("Erreur", "Le fichier de scores est introuvable.")

def afficher_aide():
    regles = "Devinez le mot avant que le pendu ne soit complet !\n7 erreurs maximum."
    messagebox.showinfo("Aide", regles)

def selectionner_difficulte():
    fenetre = tk.Toplevel(root)
    fenetre.title("Difficulté")
    def set_diff(c):
        global diff_choisie
        diff_choisie = random.choice(["Facile", "Moyen", "Difficile"]) if c == "Aléatoire" else c
        fenetre.destroy()
    tk.Button(fenetre, text="Facile", command=lambda: set_diff("Facile")).pack(pady=5)
    tk.Button(fenetre, text="Moyen", command=lambda: set_diff("Moyen")).pack(pady=5)
    tk.Button(fenetre, text="Difficile", command=lambda: set_diff("Difficile")).pack(pady=5)

# --- Lancement de la fenêtre principale ---
root = tk.Tk()
root.title("Jeu du Pendu - IN200")
root.geometry("400x500")

tk.Label(root, text="LE JEU DU PENDU", font=("Arial", 20, "bold"), pady=20).pack()
tk.Button(root, text="Nouvelle Partie", command=demarrer_partie, bg="cyan", width=20).pack(pady=10)
tk.Button(root, text="Scores", command=afficher_score, width=20).pack(pady=10)
tk.Button(root, text="Difficulté", command=selectionner_difficulte, width=20).pack(pady=10)
tk.Button(root, text="Quitter", command=root.quit, fg="red", width=20).pack(pady=10)

root.mainloop()
